# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)
print(f"Dataset ID (@id): {metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.
We'll check what record sets, fields, and column `@id`s are available in the Croissant schema.

In [ ]:
# Get all record sets with their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    # List fields and columns for each RecordSet
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name}, @id: {field.id}, DataType: {field.data_type}")
        # If columns are available in the Field (some schemas), show them
        if hasattr(field, 'columns'):
            for column in field.columns:
                print(f"      * Column Name: {column.name}, @id: {column.id}")

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis.
All entities are referenced by their `@id`. We loop through available record sets, load records, and preview columns.

In [ ]:
# Extract all the record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet with @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
For demonstration, let's select the main survey record set, filter on GAD-7 score field, normalize, and group by a demographic field.

In [ ]:
# Example: Use the main record set and mental health score field for EDA

# Choose a record set (update this to your specific record set `@id` from the overview)
survey_record_set_id = record_set_ids[0]  # Assuming the main survey record set is first
df = dataframes[survey_record_set_id]

# Print available column @ids for clarity
print("Available columns:", df.columns.tolist())

# Choose a numeric mental health field (e.g., GAD-7 total score)
numeric_field_id = None
for col in df.columns:
    if 'gad7' in col.lower() and 'score' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try another numeric field
    for col in df.columns:
        if 'phq9' in col.lower() and 'score' in col.lower():
            numeric_field_id = col
            break

if numeric_field_id is None:
    # Fallback: first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field '@id': {numeric_field_id}")

# Filter records based on the numeric field and threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization of the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping data by a demographic field (e.g., 'gender' or 'level_of_education')
group_field_id = None
for col in df.columns:
    if 'gender' in col.lower():
        group_field_id = col
        break
if not group_field_id:
    for col in df.columns:
        if 'education' in col.lower():
            group_field_id = col
            break
if not group_field_id:
    for col in df.columns:
        if 'age' in col.lower():
            group_field_id = col
            break
print(f"Grouping by field '@id': {group_field_id}")
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

In [ ]:
# Visualization: Histogram and Boxplot
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If grouping field is available, plot boxplot
if group_field_id:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=25)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded FAIR² Mental Health Survey dataset metadata and records via Croissant schema.
- Identified main record sets, fields, and relevant column `@id`s.
- Conducted basic EDA: filtered, normalized, and grouped data by demographic attributes.
- Visualized distributions and differences across groups.

This notebook serves as a reproducible template for working with Croissant datasets and exploring them with `mlcroissant` in Python.